In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import streamlit as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [19]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def scrape_udemy():
    options = uc.ChromeOptions()
    # options.add_argument('--headless') # Keep this commented to see what's happening!
    
    driver = uc.Chrome(options=options, version_main=145) # Using your version from earlier
    
    try:
        url = "https://www.udemy.com/courses/it-and-software/it-certification/"
        print("Opening Udemy... If you see a CAPTCHA, please solve it manually in the browser window.")
        driver.get(url)

        # 1. Scroll down to trigger lazy loading
        driver.execute_script("window.scrollTo(0, 1000);")
        time.sleep(3) 

        # 2. Use a more generic wait. We wait for ANY link that looks like a course title.
        wait = WebDriverWait(driver, 30)
        
        # This looks for any H3 that contains a link to a course
        titles = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'h3 > a[href*="/course/"]')))

        print(f"Found {len(titles)} potential courses!\n" + "="*50)

        for title_element in titles:
            # We find the parent container of the title to get the other info
            try:
                # Move up the DOM tree to find the whole card
                card = title_element.find_element(By.XPATH, "./ancestor::div[contains(@class, 'course-card')]")
                
                title_text = title_element.text
                
                # Try to find instructor (usually near the title)
                try:
                    instructor = card.find_element(By.CSS_SELECTOR, 'div[class*="instructor-list"]').text
                except:
                    instructor = "Unknown"

                print(f"Title: {title_text}")
                print(f"Instructor: {instructor}")
                print("-" * 30)
                
            except Exception as e:
                continue

    except Exception as e:
        print(f"An error occurred: {e}")
        # Keep the browser open so you can see if a CAPTCHA appeared
        input("Press Enter to close the browser...")
    finally:
        driver.quit()

if __name__ == "__main__":
    scrape_udemy()

Opening Udemy... If you see a CAPTCHA, please solve it manually in the browser window.
Found 20 potential courses!
Title: Ultimate AWS Certified Solutions Architect Associate 2026
Full Practice Exam | Learn Cloud Computing | Pass the AWS Certified Solutions Architect Associate Certification SAA-C03!
Instructor: Stephane Maarek | AWS Certified Cloud Practitioner,Solutions Architect,Developer
------------------------------
Title: [NEW] Ultimate AWS Certified Cloud Practitioner CLF-C02 2026
Full Practice Exam included + explanations | Learn Cloud Computing | Pass the AWS Cloud Practitioner CLF-C02 exam!
Instructor: Stephane Maarek | AWS Certified Cloud Practitioner,Solutions Architect,Developer
------------------------------
Title: CompTIA Security+ (SY0-701) Complete Course & Practice Exam
CompTIA Security+ (SY0-701) Bootcamp - Your preparation for the world's best cybersecurity certification!
Instructor: Jason Dion • 2.8 Million+ Enrollments Worldwide, Dion Training Solutions • ATO for 

In [22]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def scrape_coursera_robust():
    options = uc.ChromeOptions()
    # options.add_argument('--headless') 
    
    # Use your specific version 145
    driver = uc.Chrome(options=options, version_main=145)
    
    try:
        url = "https://www.coursera.org/search?productTypeDescription=Professional%20Certificates&sortBy=BEST_MATCH"
        print("🚀 Navigating to Coursera...")
        driver.get(url)
        driver.maximize_window()

        # Step 1: Wait for the main search results area to exist
        wait = WebDriverWait(driver, 30)
        
        # We look for the 'h3' tags specifically because you found them in your HTML snippet
        print("⏳ Waiting for course titles to render...")
        
        # This selector targets the H3 inside the specific anchor tag you shared
        titles_xpath = "//a[contains(@data-click-key, 'search_card')]//h3"
        
        # Scroll down slowly to "wake up" the page
        driver.execute_script("window.scrollTo(0, 500);")
        time.sleep(2)
        
        elements = wait.until(EC.presence_of_all_elements_located((By.XPATH, titles_xpath)))

        print(f"✅ Found {len(elements)} Professional Certificates!\n" + "="*50)

        for el in elements:
            try:
                # Get the title text
                title_text = el.text
                
                # To get the instructor/partner, we look at the parent container
                # Moving up 2 levels from the H3 usually hits the info block
                parent_text = el.find_element(By.XPATH, "./../..").text
                
                # Split the text or just print the block
                lines = parent_text.split('\n')
                partner = lines[1] if len(lines) > 1 else "Unknown Partner"

                print(f"Title:   {title_text}")
                print(f"Partner: {partner}")
                print("-" * 30)
            except:
                continue

    except Exception as e:
        print(f"❌ Error Detail: {e}")
        # This keeps the window open so you can see if there is a CAPTCHA
        input("Check the browser window for a CAPTCHA or error, then press Enter here to close...")
    finally:
        driver.quit()

if __name__ == "__main__":
    scrape_coursera_robust()

🚀 Navigating to Coursera...
⏳ Waiting for course titles to render...
✅ Found 12 Professional Certificates!
Title:   Google AI
Partner: Unknown Partner
------------------------------
Title:   Google Data Analytics
Partner: Unknown Partner
------------------------------
Title:   Google Project Management
Partner: Unknown Partner
------------------------------
Title:   Google Cybersecurity
Partner: Unknown Partner
------------------------------
Title:   Google IT Support
Partner: Unknown Partner
------------------------------
Title:   Google Digital Marketing & E-commerce
Partner: Unknown Partner
------------------------------
Title:   Google UX Design
Partner: Unknown Partner
------------------------------
Title:   IBM Generative AI Engineering
Partner: Unknown Partner
------------------------------
Title:   Google Advanced Data Analytics
Partner: Unknown Partner
------------------------------
Title:   Google IT Automation with Python
Partner: Unknown Partner
----------------------------

In [1]:
import streamlit as st
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
import pandas as pd
import time

# --- SCRAPER FUNCTION ---
def get_course_data(platform):
    options = uc.ChromeOptions()
    options.add_argument('--headless') 
    driver = uc.Chrome(options=options, version_main=145)
    
    courses = []
    try:
        if platform == "Udemy":
            driver.get("https://www.udemy.com/courses/it-and-software/it-certification/")
            time.sleep(5)
            # Using the broader selector we found earlier
            elements = driver.find_elements(By.CSS_SELECTOR, 'h3[data-purpose="course-title-url"]')
            for el in elements[:10]:
                courses.append({"Platform": "Udemy", "Title": el.text})
                
        elif platform == "Coursera":
            driver.get("https://www.coursera.org/search?productTypeDescription=Professional%20Certificates")
            time.sleep(5)
            elements = driver.find_elements(By.CSS_SELECTOR, 'h3.cds-CommonCard-title')
            for el in elements[:10]:
                courses.append({"Platform": "Coursera", "Title": el.text})
    finally:
        driver.quit()
    return courses

# --- STREAMLIT UI ---
st.set_page_config(page_title="Course Aggregator", layout="wide")

st.title("🎓 Multi-Platform Course Scraper")
st.write("Fetch real-time data from Udemy and Coursera.")

col1, col2 = st.columns(2)

with col1:
    if st.button("Scrape Udemy"):
        with st.spinner("Accessing Udemy..."):
            data = get_course_data("Udemy")
            st.session_state['udemy_data'] = data

with col2:
    if st.button("Scrape Coursera"):
        with st.spinner("Accessing Coursera..."):
            data = get_course_data("Coursera")
            st.session_state['coursera_data'] = data

# Display Results in a Table
all_data = []
if 'udemy_data' in st.session_state:
    all_data.extend(st.session_state['udemy_data'])
if 'coursera_data' in st.session_state:
    all_data.extend(st.session_state['coursera_data'])

if all_data:
    df = pd.DataFrame(all_data)
    st.divider()
    st.subheader("Results")
    st.dataframe(df, use_container_width=True)
    
    # Add a download button for the CSV
    csv = df.to_csv(index=False).encode('utf-8')
    st.download_button("Download as CSV", data=csv, file_name="courses.csv", mime="text/csv")

2026-03-09 19:00:16.776 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 19:00:16.778 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 19:00:17.464 
  command:

    streamlit run C:\Users\USER-PC\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-09 19:00:17.465 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 19:00:17.466 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 19:00:17.467 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 19:00:17.467 Thread 'MainThread': missing ScriptRunContext! This warning can